# Comprehensive Tutorial: Multiphase Galactic Wind Modeling

This tutorial demonstrates how to use the `multiphasegalacticwind` package to model multiphase galactic winds driven by stellar feedback. We'll explore a realistic wind model and create observational predictions.

## Physical Overview

The model simulates steady-state galactic winds with:
- **Hot phase**: High-temperature gas that expands adiabatically
- **Cold clouds**: Embedded clouds that exchange mass via turbulent mixing
- **Mass transfer**: Turbulent radiative mixing layers (TRMLs) mediate cloud growth/destruction

Based on Fielding & Bryan "The Structure of Multiphase Galactic Winds"

In [ ]:
# Import required packages
import os
import numpy as np
import matplotlib.pyplot as plt
from multiphasegalacticwind import (WindModel, setup_plotting_style, 
                                   plot_wind_solution, plot_column_density_distribution)

# Set up publication-quality plotting
setup_plotting_style()

# Create directory for plots
if not os.path.exists('tutorial_plots'):
    os.makedirs('tutorial_plots')
    print("Created tutorial_plots/ directory")

## Step 1: Setting Up the Wind Model

We'll create a wind model with specific physical parameters:
- **Star formation rate**: SFR = 20 M☉/yr (starburst galaxy)
- **Energy loading**: η_E = 1.0 (100% of supernova energy)
- **Hot mass loading**: η_M = 0.1 (10% of SFR goes into hot wind)
- **Cold mass loading**: η_M,cold = 0.1 (10% into cold clouds)
- **Cloud masses**: 5 species from 10 to 10⁵ M☉
- **Sonic radius**: r* = 300 pc (where flow becomes supersonic)

In [ ]:
# Create wind model with specified parameters
model = WindModel(
    # Galaxy properties
    v_circ=150.0,           # km/s, circular velocity (Milky Way-like)
    redshift=0.0,           
    
    # Wind launch properties
    SFR=20.0,               # Msun/yr, star formation rate
    eta_M=0.1,              # hot phase mass loading
    eta_M_cold=0.1,         # cold phase mass loading  
    eta_E=1.0,              # energy loading
    
    # Sonic point
    r_star_kpc=0.3,         # kpc (= 300 pc)
    
    # Cloud properties
    cloud_mass_range=(10, 1e5),    # Msun
    cloud_alpha=2.0,               # power law slope dN/dM ∝ M^-α
    N_cloud_species=5,             # 5 cloud mass bins
    T_cl=1e4,                      # K, cloud temperature
    
    # Integration settings
    r_max_kpc=100.0,        # kpc, maximum radius
    rtol=1e-6,              # relaxed tolerance for efficiency
    atol=1e-8,
)

# Display model summary
print("Model parameters:")
print(f"  SFR = {model.SFR} Msun/yr")
print(f"  η_M = {model.eta_M}")
print(f"  η_M,cold = {model.eta_M_cold}")
print(f"  η_E = {model.eta_E}")
print(f"  r_sonic = {model.r_star_kpc} kpc = {model.r_star_kpc * 1000} pc")
print(f"  N_cloud_species = {model.N_cloud_species}")
print(f"  Cloud masses = {model.M_cloud0 / 2e33} Msun")

## Step 2: Running the Wind Integration

The model solves coupled ODEs for:
- Wind velocity, density, pressure, and metallicity
- Cloud masses, velocities, and metallicities for each species

Integration uses adaptive timestepping and terminates when certain conditions are met.

In [ ]:
# Run the model
print("Running wind integration...")
solution = model.run(progress_callback='print')

# Print integration summary
print(f"\nIntegration complete!")
print(f"Maximum radius reached: {solution.r[-1]:.1f} kpc")
print(f"Final status: {solution.sol.status} - {solution.sol.message}")

## Step 3: Examining Key Results

Let's look at some key wind properties at different radii.

In [ ]:
# Extract results at specific radii
radii_of_interest = [1.0, 10.0, 50.0]  # kpc

print("Wind properties at key radii:")
print("\nr [kpc]  v [km/s]  n [cm^-3]   T [K]     M_cloud [Msun]")
print("-" * 60)

for r_kpc in radii_of_interest:
    if r_kpc <= solution.r[-1]:
        # Find closest index
        idx = np.argmin(np.abs(solution.r - r_kpc))
        
        print(f"{solution.r[idx]:6.1f}  {solution.v[idx]:8.1f}  "
              f"{solution.n[idx]:9.2e}  {solution.T[idx]:9.2e}  "
              f"{solution.M_cloud_tot[idx]:9.2e}")

# Mass loading evolution
if solution.r[-1] >= 10:
    print(f"\nMass loading at 10 kpc: {solution.mass_loading_at_10kpc:.3f}")

## Step 4: Visualizing the Wind Solution

The standard multi-panel plot shows:
1. **Top**: Velocity profiles (wind, clouds, sound speed)
2. **Middle**: Mass flux normalized by SFR
3. **Bottom**: Individual cloud masses

In [ ]:
# Create standard multi-panel plot
fig, axes = plot_wind_solution(solution, show_hot_only=True, show_clouds=True)
fig.suptitle(f'Wind Solution: SFR={model.SFR}, η_M={model.eta_M}, η_M,cold={model.eta_M_cold}')
plt.tight_layout()
plt.savefig('tutorial_plots/wind_solution.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Step 5: Calculating Velocity Distributions

For comparison with observations, we calculate dN/dv - the distribution of cloud velocities.
This is what would be observed in absorption line studies.

In [ ]:
# Calculate velocity distribution
v_cloud, dN_dv = solution.calculate_velocity_distribution(
    r_min_kpc=0.5,      # Inner radius
    r_max_kpc=50.0,     # Outer radius  
    velocity_units='km/s'
)

# Calculate statistical moments
moments = solution.calculate_velocity_moments(r_min_kpc=0.5, r_max_kpc=50.0)

print("Velocity distribution statistics:")
if 'mean' in moments:
    print(f"  Mean velocity: {moments['mean']:.1f} km/s")
    print(f"  Velocity dispersion: {moments['dispersion']:.1f} km/s")
else:
    print(f"  Velocity range: {v_cloud.min():.1f} - {v_cloud.max():.1f} km/s")

In [ ]:
# Plot velocity distribution
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(v_cloud, dN_dv, 'k-', lw=1.5)
ax.set_xlabel(r'$v$ [km s$^{-1}$]')
ax.set_ylabel(r'$dN/dv$ [(km s$^{-1}$)$^{-1}$]')
ax.set_xlim(0, 800)
ax.set_ylim(0, None)

# Add mean velocity line if available
if 'mean' in moments:
    ax.axvline(moments['mean'], color='red', ls='--', alpha=0.7, 
              label=f"Mean = {moments['mean']:.0f} km/s")
    ax.legend(frameon=False)

ax.set_title(f'Velocity Distribution (r = 0.5-50 kpc)')
plt.tight_layout()
plt.savefig('tutorial_plots/velocity_distribution.pdf', dpi=300)
plt.show()

## Step 6: Column Density Distribution

For direct comparison with absorption line observations, we calculate dN/dv in column density units [cm⁻² (km/s)⁻¹]. This shows the contribution from different cloud masses.

In [ ]:
# Calculate and plot column density distribution by species
fig, ax = plot_column_density_distribution(
    solution, 
    r_min_kpc=0.5,
    r_max_kpc=50.0,
    show_species=True,  # Show individual cloud contributions
    species_alpha=0.6,
    figsize=(6, 4.5),
    xlim=(0, 800),
    log_scale=True
)
ax.set_title('Column Density Distribution by Cloud Mass')
plt.tight_layout()
plt.savefig('tutorial_plots/column_density_by_species.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Step 7: Parameter Study

Let's explore how the cold phase mass loading (η_M,cold) affects the wind solution and velocity distribution.

In [ ]:
# Parameter study: vary eta_M_cold
eta_M_cold_values = [0.01, 0.05, 0.1, 0.2, 0.5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Store solutions for analysis
solutions = {}

for eta_M_cold in eta_M_cold_values:
    print(f"Running model with η_M,cold = {eta_M_cold}...")
    
    # Create model variant
    model_var = WindModel(
        SFR=20.0, 
        eta_M=0.1, 
        eta_M_cold=eta_M_cold,
        eta_E=1.0,
        r_star_kpc=0.3,
        cloud_mass_range=(10, 1e5),
        N_cloud_species=5,
        progress_callback=None  # Suppress progress
    )
    
    # Run and store solution
    sol_var = model_var.run()
    solutions[eta_M_cold] = sol_var
    
    # Plot velocity profile
    ax1.loglog(sol_var.r, sol_var.v, label=f'η_M,cold = {eta_M_cold}')
    
    # Calculate and plot velocity distribution
    v_cl, dN_dv = sol_var.calculate_velocity_distribution(r_min_kpc=0.5, r_max_kpc=50.0)
    if np.max(dN_dv) > 0:
        ax2.plot(v_cl, dN_dv/np.max(dN_dv), label=f'η_M,cold = {eta_M_cold}')

# Format plots
ax1.set_xlabel(r'$r$ [kpc]')
ax1.set_ylabel(r'$v$ [km/s]')
ax1.set_xlim(0.3, 100)
ax1.set_ylim(50, 3000)
ax1.legend(frameon=False, fontsize=9)
ax1.set_title('Wind Velocity Profiles')

ax2.set_xlabel(r'$v$ [km/s]')
ax2.set_ylabel(r'$dN/dv$ (normalized)')
ax2.set_xlim(0, 800)
ax2.legend(frameon=False, fontsize=9)
ax2.set_title('Velocity Distributions')

plt.tight_layout()
plt.savefig('tutorial_plots/parameter_study_eta_M_cold.pdf', dpi=300)
plt.show()

## Step 8: Analysis and Interpretation

Let's analyze the parameter study results to understand how cold mass loading affects the wind.

In [ ]:
# Analyze parameter study results
print("Effect of η_M,cold on wind properties at 10 kpc:")
print("\nη_M,cold   v [km/s]   Ṁ/SFR    M_cloud [Msun]")
print("-" * 50)

for eta_M_cold in eta_M_cold_values:
    sol = solutions[eta_M_cold]
    if sol.r[-1] >= 10:
        idx = np.argmin(np.abs(sol.r - 10.0))
        v_10 = sol.v[idx]
        mdot_10 = sol.Mdot[idx] / sol.model.SFR
        m_cloud_10 = sol.M_cloud_tot[idx]
        
        print(f"{eta_M_cold:8.2f}  {v_10:9.1f}  {mdot_10:7.3f}  {m_cloud_10:13.2e}")

## Key Takeaways

1. **Wind Acceleration**: The wind accelerates from the sonic point, reaching velocities >1000 km/s

2. **Cloud Evolution**: Smaller clouds are destroyed first, while larger clouds survive to greater distances

3. **Mass Loading**: Increasing η_M,cold reduces wind velocity but increases total mass flux

4. **Velocity Distribution**: The dN/dv distribution depends on cloud survival and acceleration

5. **Observational Predictions**: The model predicts specific velocity distributions that can be compared with absorption line observations

## Next Steps

- Try different galaxy parameters (v_circ, SFR)
- Explore the effect of turbulent mixing efficiency (f_turb0)
- Compare with observational data using the column density distributions
- Investigate different cloud mass distributions (vary cloud_alpha)

In [ ]:
# Advanced example: Custom configuration
from multiphasegalacticwind import WindConfig

# Create custom configuration with modified parameters
custom_config = WindConfig(
    f_turb0=0.2,        # Increased turbulent mixing
    drag_coeff=0.3,     # Reduced drag coefficient
    T_cl=5e3,           # Cooler clouds
)

# Create model with custom config
custom_model = WindModel(
    SFR=10.0,
    eta_M=0.05,
    eta_M_cold=0.2,
    config=custom_config
)

print("Custom model created with modified physics parameters")
print(f"  f_turb0 = {custom_model.config.f_turb0}")
print(f"  drag_coeff = {custom_model.config.drag_coeff}")
print(f"  T_cloud = {custom_model.config.T_cl} K")